# Character-level Transformer experiments

Self-contained notebook port of `src/ml_exp`, excluding `__init__.py`. Run the cells from top to bottom.

## Imports and experiment variables

In [ ]:
from datetime import datetime
from pathlib import Path
import re
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb

In [ ]:
# Hyperparameters and paths
device = "cuda" if torch.cuda.is_available() else "cpu"
block_size = 128
batch_size = 16
head_size = 32 // 4
n_embed = 32 * 4
rank_divisors = [n_embed // (2 ** i) for i in range(n_embed.bit_length() - 1, -1, -1)]
hidden_layer_size = 256
n_head = 4
causal = True
dtype = torch.float32
train_fraction = 0.9
transformer_blocks = 6
step_count = 10_000
lrate = 6e-4
loss_est_count = 100
checkin_iteration = 100
random_seed = 1337
pretrain_generation_tokens = 1_000
posttrain_generation_tokens = 10_000
dropout_rate = 0.2
eval_prompts = ["Once upon a time ", "Klein Moretti was ", "Melissa was a "]
data_source = Path("data.txt")
training_log_root = Path("training_log")

print(f"Using device: {device}")

## Class definitions

In [ ]:
class Dataset:
    def __init__(self, data_set, train=0.9, device="cpu"):
        print("Dataset loading...")
        self.device = device
        self.data_set = data_set
        self.text = Path(data_set).read_text(encoding="utf-8")
        self.vocab = sorted(set(self.text))
        self.vocab_size = len(self.vocab)
        self.data = torch.tensor(self.encode(self.text), dtype=torch.long, device=device)
        split_idx = int(train * len(self.data))
        self.train_data = self.data[:split_idx]
        self.val_data = self.data[split_idx:]
        print(
            f"Dataset loaded from {data_set}.\nVocabulary size: {self.vocab_size}.\n"
            f"Vocabulary sample: {self.vocab[:10]}\nTraining data size: {len(self.train_data)}\n"
            f"Validation data size: {len(self.val_data)}"
        )

    def encode(self, text):
        return [self.vocab.index(character) for character in text]

    def decode(self, indices):
        return "".join(self.vocab[index] for index in indices)

    def get_batch(self, split, block_size=16, batch_size=16):
        data = self.train_data if split == "train" else self.val_data
        offsets = torch.randint(0, len(data) - block_size - 1, (batch_size,))
        x = torch.stack([data[i:i + block_size] for i in offsets])
        y = torch.stack([data[i + 1:i + block_size + 1] for i in offsets])
        return x, y

In [ ]:
class SineusoidalPositionalEncoding(nn.Module):
    def __init__(self, n_embed, dtype=torch.float32, sinusoidal_base=10_000.0, max_len=5_000):
        super().__init__()
        pe = torch.zeros(max_len, n_embed, dtype=dtype)
        self.gamma = nn.Parameter(torch.ones(1, n_embed, dtype=dtype))
        position = torch.arange(0, max_len, dtype=dtype).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, n_embed, 2, dtype=dtype)
            * (-torch.log(torch.tensor(sinusoidal_base, dtype=dtype)) / n_embed)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.gamma * self.pe[:, :x.size(1), :]


class SimplePositionalEncoding(nn.Module):
    def __init__(self, n_embed, dtype=torch.float32, max_len=5_000):
        super().__init__()
        self.positional_encoding = nn.Embedding(max_len, n_embed, dtype=dtype)

    def forward(self, x):
        return x + self.positional_encoding(torch.arange(x.size(1), device=x.device))

In [ ]:
class MLP_Full(nn.Module):
    def __init__(self, n_embed, dtype=torch.float32, dropout_rate=0.1, activation=None):
        super().__init__()
        activation = nn.GELU() if activation is None else activation
        self.net = nn.Sequential(
            nn.Linear(n_embed, n_embed, dtype=dtype),
            activation,
            nn.Linear(n_embed, n_embed, dtype=dtype),
            nn.Dropout(dropout_rate),
        )

    def forward(self, x):
        return self.net(x)


class MLP_Lowrank(nn.Module):
    def __init__(self, n_embed, rank, dtype=torch.float32, dropout_rate=0.1, activation=None):
        super().__init__()
        activation = nn.GELU() if activation is None else activation
        self.net = nn.Sequential(
            nn.Linear(n_embed, rank, dtype=dtype),
            nn.Linear(rank, n_embed, dtype=dtype),
            activation,
            nn.Linear(n_embed, rank, dtype=dtype),
            nn.Linear(rank, n_embed, dtype=dtype),
            nn.Dropout(dropout_rate),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, head_size, n_embed, n_head, dtype=torch.float32, causal=True, dropout_rate=0.1):
        super().__init__()
        self.head_size = head_size
        self.n_head = n_head
        self.causal = causal
        self.qkv = nn.Linear(n_embed, 3 * n_head * head_size, bias=False, dtype=dtype)
        self.proj = nn.Linear(n_head * head_size, n_embed, dtype=dtype)
        self.dropout = nn.Dropout(dropout_rate)
        self.attention_dropout_rate = dropout_rate

    def forward(self, x, use_cache=None, position_ids=None):
        batch_size, sequence_length, _ = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(batch_size, sequence_length, self.n_head, self.head_size).transpose(1, 2)
        k = k.view(batch_size, sequence_length, self.n_head, self.head_size).transpose(1, 2)
        v = v.view(batch_size, sequence_length, self.n_head, self.head_size).transpose(1, 2)
        out = F.scaled_dot_product_attention(
            q, k, v, is_causal=self.causal,
            dropout_p=self.attention_dropout_rate if self.training else 0.0,
        )
        out = out.transpose(1, 2).reshape(batch_size, sequence_length, -1)
        return self.proj(self.dropout(out))


class SingleSelfAttentionHead(nn.Module):
    def __init__(self, initial_block_size, head_size, n_embed, dropout_rate=0.1, dtype=torch.float32, causal=True):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False, dtype=dtype)
        self.query = nn.Linear(n_embed, head_size, bias=False, dtype=dtype)
        self.value = nn.Linear(n_embed, head_size, bias=False, dtype=dtype)
        self.layer_dropout = nn.Dropout(dropout_rate)
        self.register_buffer("mask_tril", torch.tril(torch.ones(initial_block_size, initial_block_size, dtype=torch.bool)), persistent=False)
        self.causal = causal

    def forward(self, x, use_cache=None, position_ids=None):
        k, q, v = self.key(x), self.query(x), self.value(x)
        weights = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        if self.causal:
            length = weights.shape[-1]
            if length > self.mask_tril.shape[0]:
                new_size = max(length, self.mask_tril.shape[0] * 2)
                self.register_buffer("mask_tril", torch.tril(torch.ones(new_size, new_size, dtype=torch.bool, device=x.device)), persistent=False)
            weights = weights.masked_fill(~self.mask_tril[:length, :length], float("-inf"))
        return self.layer_dropout(F.softmax(weights, dim=-1)) @ v


class KV_Computer:
    def __init__(self, max_block_size, head_size, n_embed, n_head, dtype=torch.float32, device="cpu"):
        self.max_block_size = max_block_size
        self.head_size = head_size
        self.n_embed = n_embed
        self.n_head = n_head
        self.dtype = dtype
        self.device = device

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, head_size, n_embed, n_head, rank_division=1, dtype=torch.float32, causal=True, dropout_rate=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(head_size, n_embed, n_head, dtype, causal, dropout_rate)
        self.ffwdn = MLP_Full(n_embed, dtype=dtype, dropout_rate=dropout_rate, activation=nn.GELU())
        self.layer_norm1 = nn.LayerNorm(n_embed, dtype=dtype)
        self.layer_norm2 = nn.LayerNorm(n_embed, dtype=dtype)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.dropout(x)
        x = x + self.attention(self.layer_norm1(x))
        return x + self.ffwdn(self.layer_norm2(x))


class LangModel(nn.Module):
    def __init__(self, vocab_size, block_size, head_size=16, n_embed=32, transformer_blocks=1, n_head=4, hidden_layer_size=256, rank_division=1, dtype=torch.float32, causal=True, dropout_rate=0.1):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed, dtype=dtype)
        self.positional_encoding = SineusoidalPositionalEncoding(n_embed, dtype=dtype, max_len=block_size)
        self.hiddenlayer_size = hidden_layer_size
        transformer_chain = []
        for _ in range(transformer_blocks):
            transformer_chain.append(TransformerBlock(head_size, hidden_layer_size, n_head, rank_division, dtype, causal, dropout_rate))
            transformer_chain.append(SineusoidalPositionalEncoding(hidden_layer_size, dtype=dtype, max_len=block_size))
        self.transformer_blocks = nn.Sequential(
            nn.Linear(n_embed, hidden_layer_size, dtype=dtype), *transformer_chain
        )
        self.lm_head = nn.Linear(hidden_layer_size, vocab_size, dtype=dtype)

    def forward(self, idx, targets=None):
        token_embeddings = self.positional_encoding(self.token_embedding_table(idx))
        logits = self.lm_head(self.transformer_blocks(token_embeddings))
        loss = None
        if targets is not None:
            batch, length, channels = logits.shape
            loss = F.cross_entropy(logits.view(batch * length, channels), targets.view(batch * length))
        return logits, loss

    def generation(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:])
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx = torch.cat((idx, torch.multinomial(probs, num_samples=1)), dim=1)
        return idx

## Helper functions

In [ ]:
@torch.no_grad()
def estimate_loss(model, dataset):
    model.eval()
    result = {}
    for split in ("train", "val"):
        losses = torch.empty(loss_est_count, device=device)
        for index in range(loss_est_count):
            x, y = dataset.get_batch(split, block_size=block_size, batch_size=batch_size)
            _, loss = model(x, y)
            losses[index] = loss
        result[split] = losses.mean().item()
    model.train()
    return result


@torch.no_grad()
def generate_eval_outputs(model, dataset, max_new_tokens):
    outputs = []
    for prompt in eval_prompts:
        prompt_tokens = torch.tensor([dataset.encode(prompt)], dtype=torch.long, device=device)
        generated = dataset.decode(model.generation(prompt_tokens, max_new_tokens)[0].tolist())
        outputs.append({"prompt": prompt, "generation": generated})
    return outputs


def format_duration(seconds):
    seconds = max(0, round(seconds))
    hours, remainder = divmod(seconds, 3_600)
    minutes, seconds = divmod(remainder, 60)
    if hours:
        return f"{hours:d}h {minutes:02d}m {seconds:02d}s"
    if minutes:
        return f"{minutes:d}m {seconds:02d}s"
    return f"{seconds:d}s"

## Reproducibility and data loading

In [ ]:
torch.manual_seed(random_seed)
dataset = Dataset(data_source, train=train_fraction, device=device)

## Run configuration

In [ ]:
run_tag = re.sub(r"[^A-Za-z0-9._-]+", "-", input("Tag this training run: " ).strip()).strip("._-")
if not run_tag:
    raise ValueError("The run tag must not be empty.")
run_tag

## Training sweep

> This runs every configured rank divisor, writes artifacts, and logs to Weights & Biases. With the default values it is a long-running cell.

In [ ]:
for rank_division in rank_divisors:
    print(f"Rank divisor: {rank_division}, resulting rank: {n_embed // rank_division}")
    model = LangModel(
        dataset.vocab_size, block_size, head_size, n_embed=n_embed,
        transformer_blocks=transformer_blocks, n_head=n_head,
        hidden_layer_size=hidden_layer_size, rank_division=rank_division,
        dtype=dtype, causal=causal, dropout_rate=dropout_rate,
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lrate)

    run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    run_directory = training_log_root / f"{run_timestamp}_{run_tag}"
    run_directory.mkdir(parents=True, exist_ok=False)
    hyperparameters = {
        "run_timestamp": run_timestamp, "run_tag": run_tag,
        "data_source": str(data_source), "device": device, "dtype": str(dtype),
        "random_seed": random_seed, "vocab_size": dataset.vocab_size,
        "block_size": block_size, "batch_size": batch_size, "head_size": head_size,
        "n_embed": n_embed, "hidden_layer_size": hidden_layer_size, "n_head": n_head,
        "rank_division": rank_division, "causal": causal, "train_fraction": train_fraction,
        "step_count": step_count, "learning_rate": lrate,
        "optimizer": type(optimizer).__name__, "loss_est_count": loss_est_count,
        "checkpoint_interval": checkin_iteration,
        "parameter_count": sum(parameter.numel() for parameter in model.parameters()),
        "pretrain_generation_tokens": pretrain_generation_tokens,
        "posttrain_generation_tokens": posttrain_generation_tokens,
        "eval_prompts": eval_prompts, "transformer_blocks": transformer_blocks,
        "dropout_rate": dropout_rate,
    }
    with (run_directory / f"hyper_param_{run_timestamp}.txt").open("w", encoding="utf-8") as file:
        for name, value in hyperparameters.items():
            file.write(f"{name}={value}\n")

    wandb_run = wandb.init(
        project="ml-exp", name=f"{run_tag}_{run_timestamp} rank_div_{rank_division}",
        config=hyperparameters,
    )

    model.eval()
    print("Pretrain generation")
    pretrain_generations = generate_eval_outputs(model, dataset, pretrain_generation_tokens)
    for output in pretrain_generations:
        print(f"Prompt: {output['prompt']}\n{output['generation']}")
    wandb.log({"pretrain_generations": wandb.Table(
        columns=["prompt", "generation"],
        data=[[item["prompt"], item["generation"]] for item in pretrain_generations],
    )})

    model.train()
    print(f"Training loop; saving run artifacts to {run_directory}")
    loop_start = interval_start = time.perf_counter()
    previous_completed_steps = 0
    for step in range(step_count):
        x, y = dataset.get_batch("train", block_size=block_size, batch_size=batch_size)
        _, loss = model(x, y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        if step % checkin_iteration == 0 or step == step_count - 1:
            if device == "cuda":
                torch.cuda.synchronize()
            training_end = time.perf_counter()
            completed_steps = step + 1
            interval_steps = completed_steps - previous_completed_steps
            milliseconds_per_step = (training_end - interval_start) * 1_000 / interval_steps
            losses = estimate_loss(model, dataset)
            checkpoint_end = time.perf_counter()
            elapsed = checkpoint_end - loop_start
            remaining_steps = step_count - completed_steps
            eta = elapsed / completed_steps * remaining_steps
            eta_text = format_duration(eta) if completed_steps > 1 else "estimating..."
            print(
                f"step {step}: train_loss {losses['train']:.4f}, val_loss {losses['val']:.4f}, "
                f"{milliseconds_per_step:.2f} ms/step, elapsed {format_duration(elapsed)}, ETA {eta_text}"
            )
            wandb.log({
                "train_loss": losses["train"], "val_loss": losses["val"],
                "milliseconds_per_step": milliseconds_per_step,
                "elapsed_seconds": elapsed, "eta_seconds": eta,
            }, step=step)
            interval_start = checkpoint_end
            previous_completed_steps = completed_steps

    model.eval()
    print("Posttrain generation")
    posttrain_generations = generate_eval_outputs(model, dataset, posttrain_generation_tokens)
    for output in posttrain_generations:
        print(f"Prompt: {output['prompt']}\n{output['generation']}")
    wandb.log({"posttrain_generations": wandb.Table(
        columns=["prompt", "generation"],
        data=[[item["prompt"], item["generation"]] for item in posttrain_generations],
    )})
    wandb_run.finish()